# Projeto Fase I - Decolagem da Missao
## FIAP - Ciencias da Computacao

**Aluno:** Marcelo Bastianello Baldin
**RM:** 568746
**Disciplina:** Atividade Integradora - Fase I

---

Este notebook contem a simulacao completa do sistema de verificacao pre-decolagem de uma missao espacial, incluindo analise de telemetria, algoritmo de verificacao, calculos energeticos e analise assistida por IA.

## 1.1 Organizacao e Descricao da Telemetria

A telemetria da missao monitora os seguintes parametros criticos:

| Categoria | Parametros | Unidade |
|-----------|-----------|---------|
| **Termicos** | Temperatura interna e externa da nave | Graus Celsius |
| **Estruturais** | Integridade estrutural (sensor binario) | 0 = comprometida / 1 = integra |
| **Energeticos** | Nivel de carga dos bancos de energia | Percentual (%) |
| **Pressao** | Pressao dos tanques principal e reserva | bar |
| **Modulos** | Navegacao, Comunicacao, Suporte de Vida, Propulsao | operacional / falha |

Cada parametro possui uma **faixa segura** predefinida. Valores fora dessa faixa indicam risco e podem abortar a decolagem.

In [ ]:
# ============================================================
# 1.1 DADOS DE TELEMETRIA DA MISSAO
# ============================================================

# Dados coletados pelos sensores da nave
telemetria = {
    "Temperatura Interna (C)": 22.5,
    "Temperatura Externa (C)": -45.2,
    "Integridade Estrutural": 1,
    "Nivel de Energia (%)": 92,
    "Pressao Tanque Principal (bar)": 310.5,
    "Pressao Tanque Reserva (bar)": 295.0,
    "Modulo Navegacao": "operacional",
    "Modulo Comunicacao": "operacional",
    "Modulo Suporte de Vida": "operacional",
    "Modulo Propulsao": "operacional"
}

# Faixas seguras para parametros numericos
faixas_seguras = {
    "Temperatura Interna (C)": (15, 30),
    "Temperatura Externa (C)": (-60, 50),
    "Integridade Estrutural": (1, 1),
    "Nivel de Energia (%)": (80, 100),
    "Pressao Tanque Principal (bar)": (280, 350),
    "Pressao Tanque Reserva (bar)": (280, 350)
}

# Lista de modulos criticos que devem estar operacionais
modulos_criticos = [
    "Modulo Navegacao",
    "Modulo Comunicacao",
    "Modulo Suporte de Vida",
    "Modulo Propulsao"
]

print("Dados de telemetria carregados com sucesso.")

In [ ]:
# ============================================================
# EXIBICAO ORGANIZADA DOS DADOS DE TELEMETRIA
# ============================================================

print("=" * 75)
print("            RELATORIO DE TELEMETRIA PRE-DECOLAGEM")
print("=" * 75)

print(f"\n{'Parametro':<38} {'Valor':<12} {'Faixa Segura':<18} {'Status'}")
print("-" * 75)

for param, (minimo, maximo) in faixas_seguras.items():
    valor = telemetria[param]
    faixa = f"{minimo} a {maximo}"
    status = "[OK]" if minimo <= valor <= maximo else "[ALERTA]"
    print(f"{param:<38} {str(valor):<12} {faixa:<18} {status}")

print("-" * 75)
print(f"\n{'Modulos Criticos':<38} {'Status':<12} {'Verificacao'}")
print("-" * 75)

for modulo in modulos_criticos:
    status_modulo = telemetria[modulo]
    indicador = "[OK]" if status_modulo == "operacional" else "[ALERTA]"
    print(f"{modulo:<38} {status_modulo:<12} {indicador}")

print("=" * 75)

## 1.2 Algoritmo de Verificacao (Pseudocodigo e Fluxograma)

### Pseudocodigo

```
ALGORITMO VerificacaoPreDecolagem

INICIO
    DEFINIR telemetria <- ler_dados_sensores()
    DEFINIR faixas_seguras <- carregar_parametros_seguros()
    DEFINIR modulos_criticos <- ["navegacao", "comunicacao", "suporte_vida", "propulsao"]
    DEFINIR problemas <- lista_vazia

    // Verificacao dos parametros numericos
    PARA CADA parametro EM faixas_seguras:
        valor <- telemetria[parametro]
        minimo, maximo <- faixas_seguras[parametro]

        SE valor < minimo OU valor > maximo ENTAO:
            ADICIONAR "ALERTA: {parametro} fora da faixa" EM problemas
        FIM SE
    FIM PARA

    // Verificacao dos modulos criticos
    PARA CADA modulo EM modulos_criticos:
        SE telemetria[modulo] != "operacional" ENTAO:
            ADICIONAR "ALERTA: {modulo} nao operacional" EM problemas
        FIM SE
    FIM PARA

    // Decisao final
    SE problemas ESTA VAZIA ENTAO:
        EXIBIR "RESULTADO: PRONTO PARA DECOLAR"
    SENAO:
        EXIBIR "RESULTADO: DECOLAGEM ABORTADA"
        PARA CADA problema EM problemas:
            EXIBIR problema
        FIM PARA
    FIM SE
FIM
```

### Fluxograma (representacao textual)

```
    +----------------------------+
    |          INICIO            |
    |    Ler dados de telemetria |
    +-------------+--------------+
                  |
                  v
    +----------------------------+
    | PARA CADA parametro        |
    | numerico: valor esta       |
    | dentro da faixa segura?    |
    +--------+----------+--------+
             |          |
            SIM        NAO
             |          |
             v          v
          [Proximo]  [Registrar alerta]
             |          |
             +----+-----+
                  |
                  v
    +----------------------------+
    | PARA CADA modulo critico:  |
    | status == "operacional"?   |
    +--------+----------+--------+
             |          |
            SIM        NAO
             |          |
             v          v
          [Proximo]  [Registrar alerta]
             |          |
             +----+-----+
                  |
                  v
    +----------------------------+
    |    Lista de alertas        |
    |    esta vazia?             |
    +--------+----------+--------+
             |          |
            SIM        NAO
             |          |
             v          v
    +--------------+  +-----------------+
    | PRONTO PARA  |  | DECOLAGEM       |
    | DECOLAR      |  | ABORTADA        |
    +--------------+  | Listar alertas  |
                      +-----------------+
```

## 1.3 Script Python - Verificacao de Decolagem

Implementacao funcional do algoritmo de verificacao em Python:

In [ ]:
# ============================================================
# 1.3 FUNCAO DE VERIFICACAO PRE-DECOLAGEM
# ============================================================

def verificar_decolagem(telemetria, faixas_seguras, modulos_criticos):
    """
    Verifica se todos os parametros da nave estao dentro das faixas seguras
    e se todos os modulos criticos estao operacionais.

    Retorna:
        tuple: (decisao: str, problemas: list, verificacoes: list)
    """
    problemas = []
    verificacoes = []

    # Verificacao dos parametros numericos
    for parametro, (minimo, maximo) in faixas_seguras.items():
        valor = telemetria[parametro]
        dentro_faixa = minimo <= valor <= maximo

        verificacoes.append({
            "parametro": parametro,
            "valor": valor,
            "minimo": minimo,
            "maximo": maximo,
            "status": "OK" if dentro_faixa else "FALHA"
        })

        if not dentro_faixa:
            problemas.append(
                f"ALERTA: {parametro} = {valor} "
                f"(faixa segura: {minimo} a {maximo})"
            )

    # Verificacao dos modulos criticos
    for modulo in modulos_criticos:
        status = telemetria[modulo]
        operacional = status == "operacional"

        verificacoes.append({
            "parametro": modulo,
            "valor": status,
            "minimo": "operacional",
            "maximo": "operacional",
            "status": "OK" if operacional else "FALHA"
        })

        if not operacional:
            problemas.append(f"ALERTA: {modulo} esta com status '{status}'")

    # Decisao final
    if not problemas:
        decisao = "PRONTO PARA DECOLAR"
    else:
        decisao = "DECOLAGEM ABORTADA"

    return decisao, problemas, verificacoes

print("Funcao de verificacao definida com sucesso.")

In [ ]:
# ============================================================
# EXECUCAO DA VERIFICACAO PRE-DECOLAGEM - CENARIO NORMAL
# ============================================================

decisao, problemas, verificacoes = verificar_decolagem(
    telemetria, faixas_seguras, modulos_criticos
)

print("=" * 70)
print("       SISTEMA DE VERIFICACAO PRE-DECOLAGEM")
print("              Cenario: Dados Normais")
print("=" * 70)

print(f"\n{'Verificacao':<40} {'Resultado':<10}")
print("-" * 55)

for v in verificacoes:
    simbolo = "[OK]" if v["status"] == "OK" else "[FALHA]"
    print(f"  {v['parametro']:<38} {simbolo}")

print("\n" + "=" * 70)

if decisao == "PRONTO PARA DECOLAR":
    print(f"\n>>> DECISAO FINAL: {decisao}")
    print("    Todos os sistemas operacionais. Decolagem autorizada.")
else:
    print(f"\n>>> DECISAO FINAL: {decisao}")
    print("    Problemas detectados:")
    for p in problemas:
        print(f"    -> {p}")

print("\n" + "=" * 70)

### Teste com cenario de anomalia

Para validar o algoritmo, simulamos um cenario com parametros fora da faixa segura:

In [ ]:
# ============================================================
# TESTE COM CENARIO DE ANOMALIA
# ============================================================

telemetria_anomala = {
    "Temperatura Interna (C)": 35.8,       # ACIMA da faixa (max: 30)
    "Temperatura Externa (C)": -45.2,
    "Integridade Estrutural": 1,
    "Nivel de Energia (%)": 72,              # ABAIXO da faixa (min: 80)
    "Pressao Tanque Principal (bar)": 310.5,
    "Pressao Tanque Reserva (bar)": 270.0,   # ABAIXO da faixa (min: 280)
    "Modulo Navegacao": "operacional",
    "Modulo Comunicacao": "falha",            # MODULO COM FALHA
    "Modulo Suporte de Vida": "operacional",
    "Modulo Propulsao": "operacional"
}

decisao2, problemas2, verificacoes2 = verificar_decolagem(
    telemetria_anomala, faixas_seguras, modulos_criticos
)

print("=" * 70)
print("       SISTEMA DE VERIFICACAO PRE-DECOLAGEM")
print("           Cenario: Dados com Anomalias")
print("=" * 70)

print(f"\n{'Verificacao':<40} {'Resultado':<10}")
print("-" * 55)

for v in verificacoes2:
    simbolo = "[OK]" if v["status"] == "OK" else "[FALHA]"
    print(f"  {v['parametro']:<38} {simbolo}")

print("\n" + "=" * 70)

if decisao2 == "PRONTO PARA DECOLAR":
    print(f"\n>>> DECISAO FINAL: {decisao2}")
else:
    print(f"\n>>> DECISAO FINAL: {decisao2}")
    print("    Problemas detectados:")
    for p in problemas2:
        print(f"    -> {p}")

print("\n" + "=" * 70)

## 1.4 Analise Energetica

Calculo da autonomia inicial da missao, considerando:
- **Capacidade total** dos bancos de energia
- **Carga atual** no momento da verificacao
- **Consumo estimado** durante a fase de decolagem
- **Perdas energeticas** por dissipacao termica e entropia (2a Lei da Termodinamica)

Conforme estudado no Cap. 7 do material, a eficiencia energetica corresponde a razao entre a energia efetivamente aproveitada e o total de energia consumida. O rendimento global (eta) e dado por:

**eta = (Energia util / Energia total consumida) x 100**

In [ ]:
# ============================================================
# 1.4 ANALISE ENERGETICA DA MISSAO
# ============================================================

# Parametros energeticos da nave
capacidade_total_kwh = 500          # Capacidade total dos bancos de energia (kWh)
carga_atual_percent = 92            # Carga atual (%)
consumo_decolagem_kwh = 150         # Consumo estimado na decolagem (kWh)
taxa_perda_percent = 5              # Perdas energeticas - dissipacao termica (%)
consumo_orbita_kwh_h = 12           # Consumo medio em orbita (kWh/h)

# ---- CALCULOS ----

# 1. Energia disponivel atual
#    E_disponivel = Capacidade_total x (Carga_atual / 100)
energia_disponivel = capacidade_total_kwh * (carga_atual_percent / 100)

# 2. Perdas energeticas (efeito Joule / entropia)
#    Conforme a 2a Lei da Termodinamica (Cap. 7), parte da energia e
#    inevitavelmente dissipada como calor durante as conversoes energeticas.
#    Em circuitos eletricos: P = I^2 x R (energia dissipada como calor)
perdas_energeticas = energia_disponivel * (taxa_perda_percent / 100)

# 3. Energia liquida apos decolagem
#    E_liquida = E_disponivel - Consumo_decolagem - Perdas
energia_pos_decolagem = energia_disponivel - consumo_decolagem_kwh - perdas_energeticas

# 4. Autonomia estimada em orbita
#    Autonomia = E_liquida / Consumo_medio_por_hora
autonomia_horas = energia_pos_decolagem / consumo_orbita_kwh_h

# 5. Rendimento global (eta)
#    eta = (Energia util / Energia total) x 100
rendimento = ((energia_disponivel - perdas_energeticas) / energia_disponivel) * 100

# ---- RELATORIO ----

print("=" * 70)
print("            ANALISE ENERGETICA DA MISSAO")
print("=" * 70)

print(f"\nPARAMETROS DE ENTRADA:")
print(f"  Capacidade total dos bancos:     {capacidade_total_kwh} kWh")
print(f"  Carga atual:                     {carga_atual_percent}%")
print(f"  Consumo estimado (decolagem):    {consumo_decolagem_kwh} kWh")
print(f"  Taxa de perdas energeticas:      {taxa_perda_percent}%")
print(f"  Consumo medio em orbita:         {consumo_orbita_kwh_h} kWh/h")

print(f"\nCALCULOS:")
print(f"  1. Energia disponivel:           {capacidade_total_kwh} x {carga_atual_percent}% = {energia_disponivel:.1f} kWh")
print(f"  2. Perdas energeticas:           {energia_disponivel:.1f} x {taxa_perda_percent}% = {perdas_energeticas:.1f} kWh")
print(f"  3. Energia pos-decolagem:        {energia_disponivel:.1f} - {consumo_decolagem_kwh} - {perdas_energeticas:.1f} = {energia_pos_decolagem:.1f} kWh")
print(f"  4. Autonomia em orbita:          {energia_pos_decolagem:.1f} / {consumo_orbita_kwh_h} = {autonomia_horas:.1f} horas")
print(f"  5. Rendimento global (eta):      {rendimento:.1f}%")

print(f"\nRESULTADO:")
print(f"  Energia restante pos-decolagem:  {energia_pos_decolagem:.1f} kWh")
print(f"  Autonomia estimada:              {autonomia_horas:.1f} horas ({autonomia_horas/24:.1f} dias)")

if autonomia_horas >= 24:
    print(f"  Status:                          AUTONOMIA SUFICIENTE")
elif autonomia_horas >= 12:
    print(f"  Status:                          AUTONOMIA LIMITADA - monitorar")
else:
    print(f"  Status:                          AUTONOMIA INSUFICIENTE")

print("\n" + "=" * 70)

## 1.5 Analise Assistida por IA

Utilizamos uma abordagem de **classificacao** (aprendizado supervisionado, conforme Cap. 6) para categorizar cada parametro de telemetria e identificar possiveis anomalias.

O sistema classifica os dados em tres categorias:

| Classificacao | Criterio | Acao |
|:---:|---|---|
| **NORMAL** | Valor entre 15% e 85% da faixa segura | Nenhuma acao necessaria |
| **ATENCAO** | Valor proximo aos limites (0-15% ou 85-100%) | Monitoramento intensivo |
| **CRITICO** | Valor fora da faixa segura | Intervencao imediata |

Esta abordagem simula um modelo de classificacao supervisionada, onde as faixas seguras servem como dados de treinamento rotulados (labels) para definir as classes.

In [ ]:
# ============================================================
# 1.5 ANALISE ASSISTIDA POR IA
# ============================================================

def classificar_parametro(valor, minimo, maximo):
    """
    Classifica um parametro com base na sua posicao dentro da faixa segura.
    Simula um modelo de classificacao supervisionada.

    Retorna: 'NORMAL', 'ATENCAO' ou 'CRITICO'
    """
    if valor < minimo or valor > maximo:
        return "CRITICO"

    faixa = maximo - minimo
    if faixa == 0:  # Parametro binario (ex: integridade)
        return "NORMAL" if valor == minimo else "CRITICO"

    posicao_relativa = (valor - minimo) / faixa

    if posicao_relativa < 0.15 or posicao_relativa > 0.85:
        return "ATENCAO"
    else:
        return "NORMAL"


def analise_ia(telemetria, faixas_seguras, modulos_criticos):
    """
    Realiza analise assistida por IA:
    1. Classificacao dos dados
    2. Identificacao de anomalias
    3. Sugestoes de risco
    """
    classificacoes = {}
    anomalias = []
    sugestoes = []

    # 1. Classificacao dos parametros numericos
    for param, (minimo, maximo) in faixas_seguras.items():
        valor = telemetria[param]
        classe = classificar_parametro(valor, minimo, maximo)
        classificacoes[param] = {
            "valor": valor,
            "classe": classe,
            "faixa": f"{minimo} a {maximo}"
        }

        # 2. Identificacao de anomalias
        if classe == "CRITICO":
            anomalias.append(
                f"{param}: valor {valor} FORA da faixa segura ({minimo} a {maximo})"
            )
        elif classe == "ATENCAO":
            anomalias.append(
                f"{param}: valor {valor} PROXIMO ao limite da faixa segura"
            )

    # Classificacao dos modulos
    for modulo in modulos_criticos:
        status = telemetria[modulo]
        classe = "NORMAL" if status == "operacional" else "CRITICO"
        classificacoes[modulo] = {
            "valor": status,
            "classe": classe,
            "faixa": "operacional"
        }
        if classe == "CRITICO":
            anomalias.append(f"{modulo}: status '{status}' - NAO OPERACIONAL")

    # 3. Sugestoes de risco
    criticos = sum(1 for c in classificacoes.values() if c["classe"] == "CRITICO")
    atencao = sum(1 for c in classificacoes.values() if c["classe"] == "ATENCAO")

    if criticos > 0:
        sugestoes.append(f"[RISCO ALTO] {criticos} parametro(s) em estado CRITICO")
        sugestoes.append("  Recomendacao: manutencao corretiva antes de nova tentativa")

    if atencao > 0:
        sugestoes.append(f"[RISCO MEDIO] {atencao} parametro(s) em ATENCAO")
        sugestoes.append("  Recomendacao: verificar tendencia nas proximas leituras")

    if criticos == 0 and atencao == 0:
        sugestoes.append("[RISCO BAIXO] Todos os parametros em estado NORMAL")
        sugestoes.append("  Recomendacao: prosseguir com protocolo padrao de decolagem")

    # Analise de correlacao simples
    temp_interna = telemetria.get("Temperatura Interna (C)", 0)
    nivel_energia = telemetria.get("Nivel de Energia (%)", 0)
    if temp_interna > 25 and nivel_energia < 85:
        sugestoes.append("[CORRELACAO] Temperatura alta + energia baixa pode indicar")
        sugestoes.append("  sobrecarga nos sistemas de refrigeracao")

    return classificacoes, anomalias, sugestoes

print("Funcoes de analise IA definidas com sucesso.")

In [ ]:
# ============================================================
# ANALISE IA - CENARIO NORMAL
# ============================================================

classif, anom, sug = analise_ia(telemetria, faixas_seguras, modulos_criticos)

print("=" * 70)
print("       ANALISE ASSISTIDA POR IA - CENARIO NORMAL")
print("=" * 70)

print(f"\nCLASSIFICACAO DOS DADOS:")
print(f"  {'Parametro':<38} {'Valor':<12} {'Classe':<10}")
print("  " + "-" * 62)

for param, info in classif.items():
    icone = {"NORMAL": "[N]", "ATENCAO": "[A]", "CRITICO": "[C]"}[info["classe"]]
    print(f"  {param:<38} {str(info['valor']):<12} {icone} {info['classe']}")

print(f"\nANOMALIAS DETECTADAS:")
if anom:
    for a in anom:
        print(f"  -> {a}")
else:
    print("  Nenhuma anomalia detectada.")

print(f"\nSUGESTOES DE RISCO:")
for s in sug:
    print(f"  {s}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# ANALISE IA - CENARIO COM ANOMALIAS
# ============================================================

classif2, anom2, sug2 = analise_ia(
    telemetria_anomala, faixas_seguras, modulos_criticos
)

print("=" * 70)
print("     ANALISE ASSISTIDA POR IA - CENARIO COM ANOMALIAS")
print("=" * 70)

print(f"\nCLASSIFICACAO DOS DADOS:")
print(f"  {'Parametro':<38} {'Valor':<12} {'Classe':<10}")
print("  " + "-" * 62)

for param, info in classif2.items():
    icone = {"NORMAL": "[N]", "ATENCAO": "[A]", "CRITICO": "[C]"}[info["classe"]]
    print(f"  {param:<38} {str(info['valor']):<12} {icone} {info['classe']}")

print(f"\nANOMALIAS DETECTADAS:")
if anom2:
    for a in anom2:
        print(f"  -> {a}")
else:
    print("  Nenhuma anomalia detectada.")

print(f"\nSUGESTOES DE RISCO:")
for s in sug2:
    print(f"  {s}")

print("\n" + "=" * 70)

## 1.6 Reflexao Critica

### Etica e Responsabilidade

A automacao de decisoes criticas em missoes espaciais levanta questoes eticas fundamentais. Quando um algoritmo decide entre "PRONTO PARA DECOLAR" e "DECOLAGEM ABORTADA", essa decisao impacta diretamente a seguranca de vidas humanas. A responsabilidade nao recai apenas sobre o sistema automatizado, mas sobre toda a cadeia de desenvolvimento: engenheiros que definiram os parametros, cientistas que calibraram os sensores e gestores que aprovaram os limites de tolerancia.

E essencial que sistemas de verificacao como o desenvolvido neste projeto sejam transparentes e auditaveis. Cada decisao deve ser rastreavel, e os criterios utilizados devem ser publicamente documentados. A confianca em sistemas automatizados deve ser construida com base em testes rigorosos, validacao cruzada e supervisao humana permanente. Conforme estudado no Cap. 8 sobre responsabilidade social empresarial (ISO 26000), os principios de accountability e transparencia se aplicam diretamente ao desenvolvimento de sistemas criticos.

### Impacto Social da Exploracao Espacial

A exploracao espacial vai alem da fronteira cientifica: ela gera impactos sociais profundos. Tecnologias desenvolvidas para missoes espaciais frequentemente sao transferidas para aplicacoes terrestres, como sistemas de purificacao de agua, materiais avancados para a medicina e algoritmos de previsao climatica.

Ao mesmo tempo, e necessario refletir sobre a alocacao de recursos. Investimentos em exploracao espacial devem ser equilibrados com as demandas sociais terrestres, como acesso a educacao, saude e saneamento basico. A ciencia da computacao desempenha papel central nesse equilibrio, ao otimizar processos e permitir que os beneficios da exploracao espacial sejam acessiveis a sociedade como um todo.

O conceito de Triple Bottom Line (People, Planet, Profit), apresentado no Cap. 8, reforça que qualquer empreendimento tecnologico deve considerar nao apenas o lucro, mas tambem o impacto nas pessoas e no planeta.

### Sustentabilidade Tecnologica

Conforme estudado no Cap. 7 (Eficiencia Energetica), a sustentabilidade e um pilar fundamental de qualquer empreendimento tecnologico. Na exploracao espacial, a gestao eficiente da energia nao e apenas uma questao de custo, mas de sobrevivencia da missao.

Os calculos de autonomia energetica realizados neste projeto demonstram a importancia de considerar perdas por dissipacao termica (2a Lei da Termodinamica) e de dimensionar corretamente os sistemas de energia. Assim como as **smart grids** terrestres equilibram oferta e demanda de energia em tempo real (Souza et al., 2020), os sistemas de uma nave espacial precisam fazer o mesmo em condicoes muito mais restritas.

A convergencia entre engenharia eletrica, ciencia da computacao e sustentabilidade - tema central da Fase I - se manifesta diretamente na construcao de sistemas espaciais que devem ser energeticamente eficientes, confiaveis e sustentaveis a longo prazo. O principio de Green IT, discutido no Cap. 8, nos lembra que toda tecnologia deve ser concebida considerando seu ciclo de vida completo e seu impacto ambiental.

---
*Projeto desenvolvido como parte da Atividade Integradora da Fase I - Decolagem da Missao, Ciencias da Computacao - FIAP (2026)*